# Output-only identification of thermoacoustic systems
## using polynomial chaos expansions — Tutorial

This notebook reproduces the results of the supplementary material of

> Silva, C.F. et al. (2026). *Output-only identification of thermoacoustic
> systems using polynomial chaos expansions.* Combustion and Flame.

Each section references the corresponding equations in the paper (Eq. X)
and supplementary material (Eq. S-X).

---
**Outline**
1. The Langevin SDE model
2. Polynomial chaos (PC) basis — norms and tensors
3. Mean-square displacement (MSD) and diffusion coefficient *c*
4. Two-level correction for non-Gaussian envelopes
5. Parameter identification via Eqs. 10–11
6. Comparison with Kramers–Moyal reference method
7. Sensitivity study: number of samples

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ooigpc modules
from ooigpc.pc_basis import PCBasis
from ooigpc.solvers.langevin import simulate_langevin, langevin_params
from ooigpc.diffusion import compute_msd, fit_msd
from ooigpc.correction import apply_correction
from ooigpc.igpc import identify, identify_from_signal

plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

---
## 1 — The Langevin SDE model

The amplitude envelope $A(t)$ of a self-excited thermoacoustic oscillator
satisfies the Itô SDE (Eq. 1):

$$
\mathrm{d}A = \underbrace{\left(\nu A - \frac{\kappa}{8}A^3 + \frac{c}{A}\right)}_{\text{drift}} \mathrm{d}t
              + \underbrace{\sqrt{2c}}_{\text{diffusion}} \mathrm{d}W
$$

with internal coefficients $a = \nu$, $b = \kappa/8$, $c = \Gamma/(4\omega_0^2)$.

The Euler–Maruyama discretisation (Eq. 3) is:

$$
A_{n+1} = A_n + a\,\Delta t\,A_n - b\,\Delta t\,A_n^3 + \frac{c}{A_n}\Delta t
          + \sqrt{2c\,\Delta t}\;\xi_n, \quad \xi_n \sim \mathcal{N}(0,1)
$$

In [ ]:
# Physical parameters (Table 1 in the paper)
omega  = 2 * np.pi * 120   # angular frequency ω₀ = 2π × 120 Hz
nu     = 16.0              # linear growth rate ν  (s⁻¹)
kappa  = 3.4               # nonlinear coefficient κ
Gamma  = 1e7               # noise intensity Γ  (Pa²·s⁻³)

a, b, c_true = langevin_params(nu, kappa, Gamma, omega)
print(f"Internal coefficients:  a = {a}, b = {b:.5f}, c = {c_true:.4f}")

# Simulate (2 × 10⁶ steps, Δt = 5 × 10⁻⁵ s)
dt   = 5e-5
N    = 2_000_000
x    = simulate_langevin(nu=nu, kappa=kappa, Gamma=Gamma, omega=omega,
                         N=N, dt=dt, seed=0)
time = np.linspace(0, dt * N, N + 1)

print(f"Simulated {N:,} steps → x.shape = {x.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Time series (short excerpt)
i_plot = slice(0, 20_000)
axes[0].plot(time[i_plot], x[i_plot], lw=0.5, color='steelblue')
axes[0].set(xlabel='Time (s)', ylabel='A(t)', title='Amplitude envelope')

# PDF
x_pos = x[5000:][x[5000:] > 0]
axes[1].hist(x_pos, bins=120, density=True, color='steelblue', alpha=0.7,
             edgecolor='none')
axes[1].set(xlabel='A', ylabel='PDF', title='Amplitude PDF')

plt.tight_layout()
plt.show()

---
## 2 — Polynomial chaos (PC) basis

We expand $A$ in the probabilist's Hermite polynomials $\Psi_k$ (Suppl. §2):

| $k$ | $\Psi_k(\xi)$ | $\|\Psi_k\|^2 = k!$ |
|-----|--------------|---------------------|
| 0   | $1$          | $1$                 |
| 1   | $\xi$        | $1$                 |
| 2   | $\xi^2-1$    | $2$                 |
| 3   | $\xi^3-3\xi$ | $6$                 |

The gPC tensors (Eq. S-4, S-5) are:
$$C_{ijk} = \langle\Psi_i\Psi_j\Psi_k\rangle, \qquad D_{ijkl} = \langle\Psi_i\Psi_j\Psi_k\Psi_l\rangle$$

In [ ]:
basis = PCBasis(n_modes=4)
print("Norms ‖Ψₖ‖² = k!:", basis.norm)
print()
print("3rd-order tensor C[i,j,k] — key entries:")
print(f"  C[1,1,2] = ⟨ξ·ξ·(ξ²-1)⟩ = E[ξ⁴]-E[ξ²] = 3-1 = {basis.C[1,1,2]:.1f}  ✓")
print(f"  C[1,2,3] = {basis.C[1,2,3]:.1f}  (expect 6)")
print(f"  C[2,2,0] = ‖Ψ₂‖² = {basis.C[2,2,0]:.1f}  (expect 2!)")
print()
print("4th-order tensor D — key entries:")
print(f"  D[1,1,1,1] = E[ξ⁴] = {basis.D[1,1,1,1]:.1f}  (expect 3)")
print(f"  D[2,2,2,2] = E[(ξ²-1)⁴] = {basis.D[2,2,2,2]:.1f}")

---
## 3 — Mean-square displacement and diffusion coefficient *c*

The MSD (Eq. 5) is fitted by the exponential model (Eq. S-11):

$$
\text{MSD}(\tau) = \langle(A(t+\tau)-A(t))^2\rangle \approx \hat{A}\bigl(1 - e^{-\tau/\hat{T}}\bigr)
$$

giving the diffusion coefficient (Eq. 7):

$$c = \frac{\hat{A}}{2\hat{T}}$$

In [ ]:
tau_values = np.arange(2.5e-4, 0.201, 2.5e-4)

taus, msd = compute_msd(x, tau_values, dt_sim=dt)
A_hat, T_hat, msd_fit = fit_msd(taus, msd)
c_recov = A_hat / (2 * T_hat)

print(f"Â  = {A_hat:.4f}")
print(f"T̂  = {T_hat:.4f} s")
print(f"c  = Â/(2T̂) = {c_recov:.4f}  (true c = {c_true:.4f})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(taus, msd, 'rx', ms=3, label='MSD data')
axes[0].plot(taus, msd_fit, 'b-', lw=2, label=r'$\hat{A}(1-e^{-\tau/\hat{T}})$')
axes[0].set(xlabel=r'$\tau$ (s)', ylabel='MSD', title='MSD and exponential fit')
axes[0].legend()

axes[1].plot(taus, msd / taus / 2, 'rx', ms=3, label='MSD/2τ')
axes[1].plot(taus, msd_fit / taus / 2, 'b-', lw=2, label='fit')
axes[1].axhline(c_recov, color='k', ls='--', label=f'c = {c_recov:.4f}')
axes[1].set(xlabel=r'$\tau$ (s)', ylabel='MSD / (2τ)', title='Extrapolation → c')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4 — Two-level correction for non-Gaussian envelopes

The amplitude PDF is not Gaussian: the raw $\sigma_A$ from the data needs
a small correction before entering Eqs. (10)–(11).

**Region 1** (marginally stable, $\sigma_A > 0.280\,\mu$): linear interpolation
between empirical correction curves (Eqs. S-8, S-9).

**Region 2** (linearly unstable, $\sigma_A \le 0.280\,\mu$): cubic-root correction
from the skewness of the PDF (Eq. S-7).

In [ ]:
amp_positive = x[5000:][x[5000:] > 0]
corr = apply_correction(amp_positive)

print(f"μ          = {corr.mu:.4f}")
print(f"std_raw    = {corr.std_raw:.4f}")
print(f"std_correc = {corr.std_correc:.4f}")
print(f"σ (corrected) = {corr.sigma:.4f}")
print(f"Region = {corr.region}  ({'marginally stable' if corr.region==1 else 'linearly unstable'})")

---
## 5 — Parameter identification (Eqs. 10–11)

Given $\mu$, $\sigma$, and $c$, the identification formulae are:

$$
\nu = \frac{9\sigma^4 + 2\mu^2\sigma^2 - \mu^4}{2\mu^2\sigma^2(\sigma^2 - \mu^2)} \cdot c
\tag{Eq. 10}
$$

$$
\frac{\kappa}{8} = \frac{3\sigma^2 - \mu^2}{2\mu^2\sigma^2(\sigma^2 - \mu^2)} \cdot c
\tag{Eq. 11}
$$

In [ ]:
ident = identify(amp_positive, c=c_recov, omega=omega)

print("Identification results:")
print(f"  ν     = {ident.nu:.2f}  s⁻¹   (true: {nu:.1f})")
print(f"  κ     = {ident.kappa:.3f}        (true: {kappa:.2f})")
print(f"  Γ     = {ident.Gamma:.3e}  (true: {Gamma:.1e})")
print(f"  c     = {ident.c:.4f}        (true: {c_true:.4f})")

# Error
err_nu    = abs(ident.nu - nu) / abs(nu) * 100
err_kappa = abs(ident.kappa - kappa) / abs(kappa) * 100
err_Gamma = abs(ident.Gamma - Gamma) / abs(Gamma) * 100
print(f"\nRelative errors:  ν: {err_nu:.1f}%   κ: {err_kappa:.1f}%   Γ: {err_Gamma:.1f}%")

---
## 6 — Full end-to-end pipeline

The helper `identify_from_signal` combines all steps above in one call.

In [ ]:
from ooigpc.igpc import identify_from_signal

result = identify_from_signal(x, tau_values, dt_sim=dt, omega=omega)
r = result['identification']
print(f"ν = {r.nu:.2f}  κ = {r.kappa:.3f}  Γ = {r.Gamma:.3e}")

---
## 7 — Sensitivity to number of samples

The igPC method requires long signals to accurately estimate the MSD.
The following cell sweeps over $N$ and shows how the identification error decreases.

In [ ]:
N_vec = [100_000, 200_000, 500_000, 1_000_000, 2_000_000]
nu_vec, kappa_vec = [], []

for Ni in N_vec:
    xi = x[:Ni + 1]   # use the beginning of the long simulation
    try:
        out = identify_from_signal(xi, tau_values, dt_sim=dt, omega=omega)
        nu_vec.append(out['identification'].nu)
        kappa_vec.append(out['identification'].kappa)
    except Exception as e:
        nu_vec.append(np.nan)
        kappa_vec.append(np.nan)
        print(f"N={Ni}: {e}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogx(N_vec, nu_vec, 'bo-')
axes[0].axhline(nu, color='r', ls='--', label=f'true ν={nu}')
axes[0].set(xlabel='N (samples)', ylabel='ν  (s⁻¹)', title='Growth rate identification')
axes[0].legend()

axes[1].semilogx(N_vec, kappa_vec, 'bo-')
axes[1].axhline(kappa, color='r', ls='--', label=f'true κ={kappa}')
axes[1].set(xlabel='N (samples)', ylabel='κ', title='Nonlinear coefficient identification')
axes[1].legend()

plt.tight_layout()
plt.show()